# Model Context Protocol

[Model Context Protocol (MCP)](https://modelcontextprotocol.io/docs/getting-started/intro) is an open-source standard for connecting AI applications to external systems.

External systems can include:

+ Connecting and querying data sources like local files and databases.
+ Tools such as search engines and calculators.
+ Workflows, including specialized prompts.

For example, if the user asks a model to query a site like Wikipedia, it could go on the web, find the url, go on the website and find the relevant content. 

Or as another example, it could use MCP to directly read the "content" of the Wikipedia article, saving it from having to interact and sift through the "look and feel" of the Wikipedia page (i.e. it can go right to the text instead of reading all JavaScript about formatting, web layout, colors and popups)

MCP provides a standard interface for AI applications.

![](./img/06_mcp.png)

The key participants in the MCP architecture are:

+ MCP Host: The AI application that coordinates and manages one or multiple MCP clients
+ MCP Client: A component that maintains a connection to an MCP server and obtains context from an MCP server for the MCP host to use
+ MCP Server: A program that provides context to MCP clients

For example, VS Code acts as an MCP host. When VS Code connects to an MCP server, like the Sentry MCP server, the VS Code runtime instantiates an MCP client objects that maintains a connection to the Sentry MCP server.

Note that MCPs are for making calls to external knowledge sources/tools just like an API call; difference is that APIs are built for human developers to manually integrate software systems while MCP is an open standard built for AI models and autonomous agents to dynamically interact with data and tools. But as a human developer, using MCP "on the back-end" allows you to do all of this without worrying about API interfaces and security concerns (much easier to interact with and use MCP on the command line). 

# MCP Server

The library [fastmcp](https://gofastmcp.com/getting-started/welcome) allows us to create mcp servers.

The code below, also found in `./05_src/static_mcp/server.py`, instantiates an MCP server.


```python
from fastmcp import FastMCP

mcp = FastMCP("My MCP Server")

@mcp.tool
def greet(name: str) -> str:
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run(
        transport="http",
        host="localhost", 
        port=3000, 
    )
```

## Reverse Proxy

The OpenAI SDK requires an https connection to any MCP server. To run the MCP server locally, we may need a TLS certificate provided by a trusted authority. To satisfy this requirement, we use a service called [ngrok](https://ngrok.com/). Among other services, ngrok allows us to use a URL for connecting with HTTPS (rather than HTTP). It solves the issue of provisioning a TLS certificate from a trusted authority. 


# Static MCP

In a terminal with the working directory as `./05_src/`: 

+ Use `python -m static_mcp.server` to start the server.
+ **Open a new terminal window** then use `ngrok http 3000` to set up the reverse proxy. (If you run this command in the terminal where the server is open, it will do nothing.)

All calls to the URL in the environment variable `MCP_URL` will be directed to localhost:3000. `MCP_URL` is what ngrok reports after using the second command.

In [ ]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')
import os

In [5]:
from utils.clients import get_client

client = get_client()
mcp_url = f'https://{os.getenv("MCP_DOMAIN")}/mcp'

print(f'Using MCP URL: {mcp_url}')

tools = [
    {
        "type": "mcp",
        "server_label": "greeting_service",
        "server_description": "A greeting service that returns personalized greetings.",
        "server_url": mcp_url,
        "require_approval": "never",
    },
]


resp = client.responses.create(
    model="gpt-5",
    tools=tools,
    instructions="Use the greeting service to answer the question.",
    input="Hello, I am Alice.",
)

print(resp.output_text)

Using MCP URL: https://sciuroid-jackeline-overventurously.ngrok-free.dev/mcp


APIStatusError: Error code: 424 - {'error': {'message': "Error retrieving tool list from MCP server: 'greeting_service'. Http status code: 424 (Failed Dependency)", 'type': 'external_connector_error', 'param': 'tools', 'code': 'http_error'}}

# Static Weather

A more descriptive setup, but still static mcp server can be started with module `static_weather_mcp.server`. Simply stop the previous server (CTRL+C) and start the second one.

In [ ]:
from utils.clients import get_client

client = get_client()
mcp_url = f'https://{os.getenv("MCP_DOMAIN")}/mcp'

tools = [
    {
        "type": "mcp",
        "server_label": "weather_service",
        "server_description": "A weather service that returns current weather conditions.",
        "server_url": mcp_url,
        "require_approval": "never",
    },
]


resp = client.responses.create(
    model="gpt-5",
    tools=tools,
    instructions="Use the weather service to answer the question.",
    input="What is the weather currently?",
)

print(resp.output_text)